# 05 — Run inference (CUDA)

Ports `run_model.py`: `device='mps'` -> `device='cuda'`, plus inline
visualization of the annotated prediction (the original only used
`show=True`, which pops a native window — not useful from a notebook).


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2


## Config

In [ ]:
MODEL_PATH = "snacks_best.pt"
TEST_IMAGES = ["test_1.jpg", "test_2.jpg", "test_3.jpg"]  # point these at real sample frames


In [ ]:
model = YOLO(MODEL_PATH)


## Intermediary test — run against a couple of known sample images first

Before wiring this up to a live camera feed / the turntable motor, confirm
detections look sane on a handful of static frames.


In [ ]:
def run_and_show(image_path):
    results = model.predict(source=image_path, device="cuda", save=False, verbose=False)
    result = results[0]

    annotated_bgr = result.plot()  # ultralytics draws boxes/labels for us
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 6))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title(image_path)
    plt.show()

    print(f"\nFound {len(result.boxes)} objects in {image_path}!")
    for index, box in enumerate(result.boxes.xywh):
        x_center, y_center = box[0].item(), box[1].item()
        cls_id = int(result.boxes.cls[index].item())
        conf = result.boxes.conf[index].item()
        label = model.names[cls_id]
        print(f"  Object {index + 1} [{label}, conf={conf:.2f}] -> center X: {x_center:.2f}, Y: {y_center:.2f}")

    return result


for image_path in TEST_IMAGES:
    try:
        run_and_show(image_path)
    except Exception as e:
        print(f"Skipping '{image_path}': {e}")


## Single-shot inference (matches the original `run_model.py` entry point)

In [ ]:
SOURCE_IMAGE = "1"  # e.g. a webcam index string, or a path to a frame

result = run_and_show(SOURCE_IMAGE)
